# Aspectos de reviews PT-BR: treino do aluno no Kaggle

Treina o Qwen3-1.7B com LoRA para classificar 5 aspectos (entrega, produto, atendimento, preço, embalagem) em avaliações da Olist e avalia no conjunto ouro público.

O mesmo código roda numa RX 7600 de 8 GB (ROCm) e aqui, numa T4. Repositório: https://github.com/fernandomcrsilva/review-aspects-ptbr

**Antes de rodar:** Settings → Accelerator → GPU T4 e Internet ligada.

In [ ]:
!git clone -q https://github.com/fernandomcrsilva/review-aspects-ptbr
%cd review-aspects-ptbr
!pip install -q 'peft>=0.17' 'transformers>=5.0'
!python src/get_data.py

## Baseline clássico: TF-IDF + regressão logística

In [ ]:
!python src/evaluate.py --name tfidf-haiku --tfidf data/aspectos/silver

## Aluno sem ajuste (zero-shot, prompt completo)

In [ ]:
!python src/evaluate.py --name aluno-zero-shot

## LoRA no prata do Claude Haiku

Na T4 o modelo roda em fp32 (a T4 não tem bf16 nativo).

In [ ]:
!python src/train_lora.py --silver data/aspectos/silver --out runs/aluno-haiku --batch 8 --accum 2

In [ ]:
!python src/evaluate.py --name aluno-haiku --adapter runs/aluno-haiku

## Resultados

In [ ]:
import json
# Só os modelos rodados neste notebook (results/ também traz os resultados da RX 7600, que vieram com o repositório)
for n in ['tfidf-haiku', 'aluno-zero-shot', 'aluno-haiku']:
    r = json.load(open(f'results/{n}.json'))
    print(f"{n:18} F1 {r['macro']:.3f} | pede ação {r['acao_recall']:.1%} | JSON válido {r['json_valido']:.0%} | {r['reviews_por_segundo']:.1f} avaliações/s")